# RecVAE on synthetic data

This notebook is a self-contained walk-through of the RecVAE pipeline 
for a reader with a PyTorch background who has no clinical-imaging 
dataset on hand. Every step runs on toy 3D-plus-time volumes produced 
by `recvae.synthetic_cohort`, so nothing here depends on access to any 
external data archive.

What we do, top to bottom:

1. Generate a small synthetic cohort with two subgroups (control and 
   a damped "AD-like" subgroup).
2. Train the RecVAE on those volumes for a handful of epochs.
3. Generate a fresh held-out cohort with a different RNG seed and 
   score it with `evaluate_held_out`, which re-fits *only* the 
   subject-specific `z_s` while keeping everything else frozen.
4. Run a linear probe on the model's latent states to separate the 
   two cohorts, and compare against a PCA baseline on the raw volumes.

Runtime: ~5 minutes on CPU, faster on GPU. The model architecture is 
fixed at the canonical `(91, 109, 91)` spatial shape; we shrink only 
`T` (number of timepoints) and `N` (cohort size) for speed.

## Setup

If running on Colab, run the next cell. If running locally with `pip install -e .` already done, skip the next cell.

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("/content/VAE-fMRI-Alzheimer"):
        rc = os.system("git clone https://github.com/YuZh98/VAE-fMRI-Alzheimer.git /content/VAE-fMRI-Alzheimer")
        if rc != 0:
            raise RuntimeError(f"git clone failed (exit {rc}); check network/proxy.")
    os.chdir("/content/VAE-fMRI-Alzheimer")
    rc = os.system("pip install -e .")
    if rc != 0:
        raise RuntimeError(f"pip install -e . failed (exit {rc}); check the install log above.")
    sys.path.insert(0, "/content/VAE-fMRI-Alzheimer")
else:
    # local: ensure repo root is on sys.path even when this notebook is
    # opened from notebooks/
    import pathlib
    here = pathlib.Path.cwd()
    for candidate in (here, *here.parents):
        if (candidate / "recvae").is_dir():
            sys.path.insert(0, str(candidate))
            break

In [ ]:
import numpy as np
import torch

from recvae import (
    Config,
    DeviceDataLoader,
    FMRIDataset,
    RecVAEModel,
    build_dataloader,
    evaluate_held_out,
    fit,
    get_default_device,
    normalize_per_subject,
    set_seed,
    synthetic_cohort,
)

# matplotlib is optional; the visualization cell handles its absence.
try:
    import matplotlib.pyplot as plt
    _HAVE_PLT = True
except Exception:
    _HAVE_PLT = False

In [ ]:
set_seed(2022)
device = get_default_device()
# The model has known quirks on the MPS backend (in particular .item()
# inside the training loop occasionally indexes oddly). For a fully
# reliable demo, fall back to CPU on MPS. On CUDA we leave the device alone.
if device.type == 'mps':
    print('MPS detected. Using CPU for this demo (model has known MPS quirks; see Lesson 10).')
    device = torch.device('cpu')
print('device:', device)

## Generate a synthetic cohort

`synthetic_cohort` is a deterministic generator (see 
`recvae/data.py`). The key parameters:

- `n_cn`, `n_ad` — number of subjects in each subgroup. Labels are 
  emitted in the order `[CN, ..., CN, AD, ..., AD]`.
- `T` — number of timepoints per subject. We pick `T=4` because the 
  model's `tol_time` defaults to 4 in this demo's `Config` — keeps the 
  rollout cheap on CPU.
- `spatial` — defaults to `(91, 109, 91)`, the encoder/decoder's 
  hardcoded shape. **Do not change this** unless you also change the 
  model layers.
- `cohort_effect` — fractional damping applied to a hemispheric slab 
  of the AD-subgroup spatial pattern. Crude proxy for atrophy.
- `seed` — RNG seed. Same seed gives identical output; different 
  seeds give statistically independent draws.

In [ ]:
volumes, labels = synthetic_cohort(n_cn=4, n_ad=4, T=4, seed=2022)
print('volumes shape :', tuple(volumes.shape))
print('labels        :', labels.tolist(), '  # 0=CN, 1=AD')

## Visualize one slice per cohort

Not load-bearing for the rest of the notebook — if matplotlib is not 
available, this cell is a no-op and the pipeline still runs.

In [ ]:
if _HAVE_PLT:
    try:
        cn_idx, ad_idx = 0, 4  # first CN, first AD in the [CN..., AD...] layout
        z_mid = volumes.shape[4] // 2
        fig, axes = plt.subplots(1, 2, figsize=(7, 3))
        axes[0].imshow(volumes[cn_idx, 0, :, :, z_mid, 0].numpy().T, cmap='gray', origin='lower')
        axes[0].set_title(f'CN subject (idx={cn_idx}), z={z_mid}, t=0')
        axes[1].imshow(volumes[ad_idx, 0, :, :, z_mid, 0].numpy().T, cmap='gray', origin='lower')
        axes[1].set_title(f'AD subject (idx={ad_idx}), z={z_mid}, t=0')
        for ax in axes:
            ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print('plotting skipped:', e)
else:
    print('matplotlib not available; skipping visualization.')

## Normalize per subject

Min-max scale each subject's volume to `[-1, 1]` using that subject's 
own extrema. This matches the canonical preprocessing. See Lesson 09 
for the trade-offs (it destroys inter-subject intensity scale).

In [ ]:
volumes, _vmax, _vmin = normalize_per_subject(volumes)
print('after norm: min=%.3f, max=%.3f' % (volumes.min().item(), volumes.max().item()))

## Build DataLoader and model

`FMRIDataset` yields `(volume, index)` pairs; the integer index is 
how the training step looks up `model.z_vectors[idx]`. 
`DeviceDataLoader` is a thin wrapper that moves each batch to the 
target device on iteration. 
`Config(tol_time=4, learning_rate=1e-5)` matches Lesson 15's settings 
for the same reason: at the production default `lr=1e-6` the loss 
barely moves in three epochs.

In [ ]:
cfg = Config(tol_time=4, learning_rate=1e-5)
ds = FMRIDataset(volumes)
dl = build_dataloader(ds, batch_size=2, shuffle=True, seed=2022)
dl = DeviceDataLoader(dl, device)

model = RecVAEModel(train_size=len(ds), cfg=cfg).to(device)
h0 = torch.zeros(1, cfg.latent_dim, device=device)

print('train_size       :', len(ds))
print('batches/epoch    :', len(dl))
print('latent_dim       :', cfg.latent_dim)
print('# parameters     :', sum(p.numel() for p in model.parameters()))

## Train

Three epochs is enough to see the loss move and to exercise the 
alternating optimization (SGD on the nets and `z_vectors`, plus a 
closed-form ridge update on `F_mat` once per epoch). On real data 
you would train for hundreds of epochs.

In [ ]:
result = fit(model, dl, h0, cfg=cfg, epochs=3)
history = result['train_loss_history']
print('loss history:', history)

## Held-out evaluation

We generate a brand-new cohort with a *different* RNG seed. The two 
draws are statistically independent samples from the same generative 
process, so the held-out cohort is the right thing to score on.

`evaluate_held_out` freezes the encoder, decoder, inference head, 
`F_mat`, and the training `z_vectors`, then fits a fresh `z_test` of 
shape `(N_test, latent_dim)` by plain SGD against the reconstruction 
MSE. The function snapshots and restores every `requires_grad` flag, 
so calling it has no side effects on the model.

In [ ]:
vol_test, lab_test = synthetic_cohort(n_cn=2, n_ad=2, T=4, seed=4242)
vol_test, _, _ = normalize_per_subject(vol_test)
print('test volumes :', tuple(vol_test.shape))
print('test labels  :', lab_test.tolist())

eval_out = evaluate_held_out(
    model,
    vol_test.to(device),
    h0,
    inner_steps=20,
    inner_lr=1e-3,
)
print('held-out reconstruction MSE: %.6f' % eval_out['recon_mse'])
print('z_test shape :', tuple(eval_out['z_test'].shape))
print('h_test shape :', tuple(eval_out['h_test'].shape))

## Linear probe: CN vs AD on latent representations

To check whether the model picked up the cohort-localized damping 
introduced by `synthetic_cohort`, we extract the per-timepoint 
latent states `h_t` from a full forward pass over the training 
cohort, average across time, and fit a logistic regression on those 
vectors against the cohort labels. As a sanity baseline we do the 
same with the top-K PCA components of the raw flattened volumes.

Caveat: N=8 is tiny. Training accuracy here is a *signal-presence* 
check, not a generalization claim. Treat the numbers as "is the 
structure in the latents at all?" rather than as a benchmark.

In [ ]:
# (a) Extract average latent state h_mean for each training subject.
#     `forward` expects h_0 of shape (B, latent_dim) — broadcast h0 to
#     batch size N before the call.
model.eval()
with torch.no_grad():
    which = torch.arange(len(ds), device=device).long()
    h_batch = h0.expand(len(ds), -1)
    out = model(volumes.to(device), h_batch, which)
    h_mean = out.h.mean(dim=1).cpu().numpy()  # (N, latent_dim)
y = labels.numpy()
print('h_mean shape :', h_mean.shape, '  labels:', y.tolist())

# (b) Logistic regression on h_mean. sklearn is NOT a hard dep — fall
#     back to a simple threshold-on-first-discriminant rule if absent.
def _fit_logreg(X, y):
    try:
        from sklearn.linear_model import LogisticRegression
        clf = LogisticRegression(max_iter=200)
        clf.fit(X, y)
        return clf.score(X, y), 'sklearn LogisticRegression'
    except ImportError:
        # Hand-rolled fallback: project onto the mean-difference axis,
        # threshold at the midpoint. Equivalent to a Fisher-style linear
        # classifier under equal covariances.
        Xm0 = X[y == 0].mean(axis=0)
        Xm1 = X[y == 1].mean(axis=0)
        w = Xm1 - Xm0
        s = X @ w
        thr = 0.5 * (Xm0 @ w + Xm1 @ w)
        pred = (s > thr).astype(int)
        return float((pred == y).mean()), 'mean-difference threshold (no sklearn)'

acc_h, how_h = _fit_logreg(h_mean, y)
print(f'latent probe  : train acc = {acc_h:.3f}   ({how_h})')

# (c) PCA baseline on the raw flattened volumes. Same caveat about N=8.
X_flat = volumes.reshape(volumes.shape[0], -1).numpy()
X_centered = X_flat - X_flat.mean(axis=0, keepdims=True)
K = min(4, X_centered.shape[0] - 1)  # cap components at N-1
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
pca_feats = (U[:, :K] * S[:K])  # (N, K) PC scores
acc_pca, how_pca = _fit_logreg(pca_feats, y)
print(f'PCA baseline  : train acc = {acc_pca:.3f}  (K={K}, {how_pca})')

## Next steps

- `tutorials/` — the lesson series this notebook condenses. Lesson 15 
  walks through the training loop alone; Lesson 18 mirrors this 
  notebook as a single `.py` script.
- `examples/` — longer training runs and analysis scripts (where 
  present).
- `docs/background/` — math background on the model and the 
  alternating optimization (to be added).
- `recvae/evaluation.py` — read `evaluate_held_out` end to end; the 
  inner-loop rewrite of `forward` is a useful pattern when you need 
  gradients to flow into an external tensor.
- Try `cohort_effect=0.0` in `synthetic_cohort`. The CN and AD draws 
  become statistically identical and the linear probe should drop 
  to chance — a nice falsifiable check that the probe is reading 
  the cohort signal, not noise.